In [4]:
!pip install pandas

  Using cached tzdata-2025.3-py2.py3-none-any.whl.metadata (1.4 kB)
   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   ---------------- ----------------------- 3.9/9.7 MB 28.0 MB/s eta 0:00:01
   ---------------------------------------- 9.7/9.7 MB 46.2 MB/s eta 0:00:00
   ---------------------------------------- 0.0/12.3 MB ? eta -:--:--
   -------------- ------------------------- 4.5/12.3 MB 22.6 MB/s eta 0:00:01
   --------------------- ------------------ 6.6/12.3 MB 15.9 MB/s eta 0:00:01
   ------------------------------------- -- 11.5/12.3 MB 18.4 MB/s eta 0:00:01
   ---------------------------------------- 12.3/12.3 MB 16.5 MB/s eta 0:00:00
Using cached tzdata-2025.3-py2.py3-none-any.whl (348 kB)



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import json
import pandas as pd


with open("../commentMyanimelist/comment.json", "r", encoding="utf-8") as f:
    comment_data = json.load(f)

with open("../commentYoutube/Youtube_comments_cleaned_final.json", "r", encoding="utf-8") as f:
    youtube_data = json.load(f)

with open("../commentAnilist/clean_comment_anilst_final.json", "r", encoding="utf-8") as f:
    pro_data = json.load(f)

with open("../animelist/anime_2025_by_season.json", "r", encoding="utf-8") as f:
    anime_data = json.load(f)


df_comment = pd.DataFrame(comment_data).rename(columns={
    "title": "anime_name",
    "text_c": "comment",
    "uidpost": "malID"
})

df_youtube = pd.DataFrame(youtube_data)

df_pro = pd.DataFrame(pro_data).rename(columns={
    "mal_id": "malID",
    "anime": "anime_name",
    "user": "user_c",
    "text": "comment"
})

cols = ["malID", "anime_name", "user_c", "comment"]
df_comment = df_comment[cols]
df_youtube = df_youtube[cols]
df_pro = df_pro[cols]


df_all = pd.concat([df_comment, df_youtube, df_pro], ignore_index=True)

df_all['malID'] = df_all['malID'].astype(str)

print(f"Total rows before merge: {len(df_all)}") 

anime_list = []
for season, animes in anime_data["seasons"].items():
    for anime in animes:
        genres = [g["name"] for g in anime.get("genres", [])]
        anime_list.append({
            "malID": str(anime["mal_id"]), 
            "Genres": ", ".join(genres),
            "season":season
        })

df_genres = pd.DataFrame(anime_list)


df_genres = df_genres.drop_duplicates(subset=['malID'], keep='first')


df_all = df_all.merge(df_genres, on="malID", how="left")



Total rows before merge: 80076


In [ ]:

if 'ID' in df_all.columns:
    df_all = df_all.drop(columns=['ID'])
df_all.insert(0, "ID", range(1, len(df_all) + 1))

print("--- Info ---")
print(df_all.info())
print(f"Final row count: {len(df_all)}") 

display(df_all.head())

--- Info ---
<class 'pandas.DataFrame'>
RangeIndex: 80076 entries, 0 to 80075
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   ID          80076 non-null  int64
 1   malID       80076 non-null  str  
 2   anime_name  80076 non-null  str  
 3   user_c      80076 non-null  str  
 4   comment     80076 non-null  str  
 5   Genres      80076 non-null  str  
 6   season      80076 non-null  str  
dtypes: int64(1), str(6)
memory usage: 4.3 MB
None
Final row count: 80076


,ID,malID,anime_name,user_c,comment,Genres,season
0,1,58567,Ore dake Level Up na Ken Season 2: Arise from ...,Tkit,This is genuinely the most pathetic new mainst...,"Action, Adventure, Fantasy",winter
1,2,58567,Ore dake Level Up na Ken Season 2: Arise from ...,Marinate1016,No amount of memes or social media posts could...,"Action, Adventure, Fantasy",winter
2,3,58567,Ore dake Level Up na Ken Season 2: Arise from ...,keirashii,This review contains spoilers. If we were to f...,"Action, Adventure, Fantasy",winter
3,4,58567,Ore dake Level Up na Ken Season 2: Arise from ...,Bardwyne,"Just as the last season, Solo Leveling s2 has ...","Action, Adventure, Fantasy",winter
4,5,58567,Ore dake Level Up na Ken Season 2: Arise from ...,ShishiKami,It's honestly just as bad as the previous seas...,"Action, Adventure, Fantasy",winter


In [ ]:


df_missing_genres = df_all[df_all["Genres"].isna()| (df_all["Genres"].str.strip() == "")]

missing_count = len(df_missing_genres)
print(f"จำนวนคอมเมนต์ที่หา Genres ไม่เจอ: {missing_count} แถว")

missing_malids = df_missing_genres["malID"].unique()
print(f"จำนวน malID ที่ข้อมูล Genres ขาดไป: {len(missing_malids)} รายการ")

if missing_count > 0:
    print("\nตัวอย่างอนิเมะที่ไม่มีข้อมูล Genres:")
    print(df_missing_genres[["malID", "anime_name"]].drop_duplicates().head(10))

print("ก่อนลบ:", len(df_all))

df_all = df_all[~(df_all["Genres"].isna() | (df_all["Genres"].str.strip() == ""))]

print("หลังลบ:", len(df_all))

จำนวนคอมเมนต์ที่หา Genres ไม่เจอ: 13352 แถว
จำนวน malID ที่ข้อมูล Genres ขาดไป: 206 รายการ

ตัวอย่างอนิเมะที่ไม่มีข้อมูล Genres:
       malID                                         anime_name
934    59419  Project Sekai Movie: Kowareta Sekai to Utaenai...
1099   61080                                               Hebi
2402   59199                                      Classic★Stars
2439   61520                                      NinKoro Dance
4147   61997               Sugar Sugar Rune: Les deux sorcières
4152   62401                                    Ashita no Arika
4153   62052                                Muri Muri Shinkaron
8419   59419  project sekai movie: kowareta sekai to utaenai...
9908   60645  boku no hero academia the movie 4: you're next...
12716  60045      the idolm@ster shiny colors 2nd season: shhis
ก่อนลบ: 80076
หลังลบ: 66724


In [ ]:
print("--- Final Info ---")
print(df_all.info())
print(f"Final row count: {len(df_all)}") 

display(df_all.tail())

df_all.to_json(
    "anime_comments_all.json",
    orient="records",
    force_ascii=False,
    indent=4
)

--- Final Info ---
<class 'pandas.DataFrame'>
Index: 66724 entries, 0 to 80075
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   ID          66724 non-null  int64
 1   malID       66724 non-null  str  
 2   anime_name  66724 non-null  str  
 3   user_c      66724 non-null  str  
 4   comment     66724 non-null  str  
 5   Genres      66724 non-null  str  
 6   season      66724 non-null  str  
dtypes: int64(1), str(6)
memory usage: 4.1 MB
None
Final row count: 66724


,ID,malID,anime_name,user_c,comment,Genres,season
80071,80072,62144,"Potion, Wagami wo Tasukeru",Shad0wq,This is pretty nice anime Animation is obvious...,Fantasy,fall
80072,80073,62392,Jujutsu Kaisen: Shibuya Jihen Tokubetsu Henshu...,NepNep,I think that s for Recuts as per the submissio...,"Action, Supernatural",fall
80073,80074,62392,Jujutsu Kaisen: Shibuya Jihen Tokubetsu Henshu...,KillerTacos,This movie was such a waste of time man The fi...,"Action, Supernatural",fall
80074,80075,62392,Jujutsu Kaisen: Shibuya Jihen Tokubetsu Henshu...,AshMesa,I saw it there was Season 3 Episode 1 2 conten...,"Action, Supernatural",fall
80075,80076,62392,Jujutsu Kaisen: Shibuya Jihen Tokubetsu Henshu...,BloodHedgehog,Do you need to watch this before the new season,"Action, Supernatural",fall
